# Interpretation Assessment Demo, IMDb Engagement Signals

Purpose: demonstrate how interpretation is assessed quantitatively (not thematically) by fitting a simple regression model and evaluating effect size, statistical significance, and explained variance. The visual deliverable is a coefficient plot with confidence intervals.

In [ ]:
# ------------------------------------------------------------
# NOTEBOOK SETUP
# ------------------------------------------------------------
# This notebook reads from an existing SQLite database (imdb_data.db),
# pulls a modeling dataset with one row per title, then fits a simple
# regression model and generates a visual coefficient plot.
#
# Requirements:
#   pandas, numpy, matplotlib, statsmodels

from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.formula.api as smf

# Make display a little easier to read in Jupyter.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

DB_PATH = Path("imdb_data.db")

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Could not find the database file at: {DB_PATH.resolve()}\n"
        "Update DB_PATH to the correct location of imdb_data.db."
    )

print(f"Using database: {DB_PATH.resolve()}")

Using database: C:\Users\JanMc\Dropbox\Education\_GitHub_coursework\janmcconnellCityU-coursework\DS687_CAPSTONE\imdb_data.db


In [ ]:
# ------------------------------------------------------------
# DATABASE INSPECTION
# ------------------------------------------------------------
# This cell lists tables and views so you can pick the correct source.
# ------------------------------------------------------------

with sqlite3.connect(DB_PATH) as conn:
    objects = pd.read_sql_query(
        """
        SELECT name, type
        FROM sqlite_master
        WHERE type IN ('table','view')
        ORDER BY type, name;
        """,
        conn
    )

display(objects)

In [ ]:
# ------------------------------------------------------------
# MODELING QUERY
# ------------------------------------------------------------
# Replace the SQL below with the simplest query that returns one row per title,
# including averageRating, numVotes, startYear, and a genre field.
#
# TIP:
#   Start with your consolidated analytical table/view if you have one.
#
# EXAMPLE (Approach A):
#   FROM title_analytics
#
# EXAMPLE (Approach B):
#   JOIN title_basics to title_ratings on tconst
#
# ------------------------------------------------------------

SQL_MODEL = """
-- TODO: Replace this query with the correct source table or join query in YOUR db.
-- This is a placeholder example that you will edit.

SELECT
    tb.tconst                      AS title_id,
    tr.averageRating               AS averageRating,
    tr.numVotes                    AS numVotes,
    tb.startYear                   AS startYear,
    tb.genres                      AS genres_raw
FROM title_basics tb
JOIN title_ratings tr
    ON tb.tconst = tr.tconst
WHERE tr.numVotes IS NOT NULL
  AND tr.averageRating IS NOT NULL
  AND tb.startYear IS NOT NULL
  AND tb.genres IS NOT NULL
;
"""

# ------------------------------------------------------------
# RUN QUERY
# ------------------------------------------------------------
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query(SQL_MODEL, conn)

print("Rows:", len(df))
display(df.head(10))

In [ ]:
# ------------------------------------------------------------
# CLEANING AND FEATURE ENGINEERING
# ------------------------------------------------------------
# Keep this intentionally simple. The goal is a defensible demo:
#   production attribute(s) -> engagement metric
# and interpretation assessed via model coefficients and plots.
# ------------------------------------------------------------

df = df.copy()

# Convert to numeric, forcing invalid values to NaN so we can drop them cleanly.
df["averageRating"] = pd.to_numeric(df["averageRating"], errors="coerce")
df["numVotes"] = pd.to_numeric(df["numVotes"], errors="coerce")
df["startYear"] = pd.to_numeric(df["startYear"], errors="coerce")

# Drop rows missing critical fields.
df = df.dropna(subset=["averageRating", "numVotes", "startYear"])

# Log transform votes:
# - log1p handles 0 safely: log(1 + x)
# - vote counts are typically highly skewed, this makes regression more stable.
df["log_numVotes"] = np.log1p(df["numVotes"])

# Primary genre:
# IMDb genres is often like: "Comedy,Romance"
# For a simple demo, take the first genre as primary.
# This is not "the truth", it is a practical first pass for modeling.
if "genres_raw" in df.columns:
    df["primaryGenre"] = (
        df["genres_raw"]
        .astype(str)
        .str.split(",", expand=False)
        .str[0]
        .str.strip()
    )
else:
    # If your query already returns primaryGenre, this keeps it consistent.
    # You can delete this branch once your SQL returns primaryGenre explicitly.
    df["primaryGenre"] = df.get("primaryGenre", "Unknown")

# Filter out junk or empty genres.
df["primaryGenre"] = df["primaryGenre"].replace({"\\N": np.nan, "": np.nan})
df = df.dropna(subset=["primaryGenre"])

# Optional: keep only reasonably frequent genres so the coefficient plot is readable.
min_count = 500  # adjust as needed for your dataset size
genre_counts = df["primaryGenre"].value_counts()
keep_genres = genre_counts[genre_counts >= min_count].index

df = df[df["primaryGenre"].isin(keep_genres)].copy()

print("Rows after cleaning:", len(df))
print("Genres kept:", df["primaryGenre"].nunique())
display(df[["title_id", "averageRating", "numVotes", "log_numVotes", "startYear", "primaryGenre"]].head(10))

In [ ]:
# ------------------------------------------------------------
# REGRESSION MODEL
# ------------------------------------------------------------
# Using statsmodels formula interface:
#   C(primaryGenre) tells statsmodels to treat genre as categorical and create dummies.
#
# startYear is included as a simple numeric control to reduce obvious period effects.
# ------------------------------------------------------------

model = smf.ols(
    formula="log_numVotes ~ C(primaryGenre) + startYear",
    data=df
).fit()

# Model summary is your quantitative assessment artifact.
# It includes coefficients, p-values, R-squared, etc.
print(model.summary())

In [ ]:
# ------------------------------------------------------------
# COEFFICIENT PLOT
# ------------------------------------------------------------
# We will plot genre coefficients only.
#
# Notes:
# - statsmodels sets one genre as the baseline (reference category).
# - Coefficients shown are differences from that baseline.
# - Confidence intervals help communicate uncertainty visually.
# ------------------------------------------------------------

params = model.params
conf = model.conf_int()
conf.columns = ["ci_low", "ci_high"]

coef_df = pd.concat([params.rename("coef"), conf], axis=1)

# Keep only genre terms: they look like "C(primaryGenre)[T.Comedy]"
genre_rows = coef_df.index.str.startswith("C(primaryGenre)[T.")
genre_coef = coef_df.loc[genre_rows].copy()

# Clean up labels for plotting.
genre_coef["genre"] = (
    genre_coef.index
    .str.replace("C(primaryGenre)[T.", "", regex=False)
    .str.replace("]", "", regex=False)
)

# Sort by coefficient magnitude so plot is readable.
genre_coef = genre_coef.sort_values("coef")

# Plot
plt.figure(figsize=(10, 6))
y = np.arange(len(genre_coef))

plt.hlines(y=y, xmin=genre_coef["ci_low"], xmax=genre_coef["ci_high"])
plt.plot(genre_coef["coef"], y, marker="o", linestyle="")

plt.yticks(y, genre_coef["genre"])
plt.axvline(0, linewidth=1)

plt.xlabel("Estimated effect on log(numVotes), relative to baseline genre")
plt.ylabel("Primary genre")
plt.title("Genre Effects on Engagement (log vote count): Coefficients with 95% CI")

plt.tight_layout()
plt.show()

# Quick model fit numbers you can paste into a slide
print("R-squared:", round(model.rsquared, 4))
print("Adjusted R-squared:", round(model.rsquared_adj, 4))
print("N:", int(model.nobs))

In [ ]:
# ------------------------------------------------------------
# OPTIONAL SECOND MODEL: AVERAGE RATING
# ------------------------------------------------------------
# Keep the same predictors to make the comparison straightforward.
# ------------------------------------------------------------

rating_model = smf.ols(
    formula="averageRating ~ C(primaryGenre) + startYear",
    data=df
).fit()

print(rating_model.summary())

params2 = rating_model.params
conf2 = rating_model.conf_int()
conf2.columns = ["ci_low", "ci_high"]
coef2_df = pd.concat([params2.rename("coef"), conf2], axis=1)

genre_rows2 = coef2_df.index.str.startswith("C(primaryGenre)[T.")
genre_coef2 = coef2_df.loc[genre_rows2].copy()
genre_coef2["genre"] = (
    genre_coef2.index
    .str.replace("C(primaryGenre)[T.", "", regex=False)
    .str.replace("]", "", regex=False)
)
genre_coef2 = genre_coef2.sort_values("coef")

plt.figure(figsize=(10, 6))
y2 = np.arange(len(genre_coef2))

plt.hlines(y=y2, xmin=genre_coef2["ci_low"], xmax=genre_coef2["ci_high"])
plt.plot(genre_coef2["coef"], y2, marker="o", linestyle="")

plt.yticks(y2, genre_coef2["genre"])
plt.axvline(0, linewidth=1)

plt.xlabel("Estimated effect on averageRating, relative to baseline genre")
plt.ylabel("Primary genre")
plt.title("Genre Effects on Average Rating: Coefficients with 95% CI")

plt.tight_layout()
plt.show()

print("R-squared:", round(rating_model.rsquared, 4))
print("Adjusted R-squared:", round(rating_model.rsquared_adj, 4))
print("N:", int(rating_model.nobs))